# Chapter 24
## Synchronization by Fast Recurrent Excitation
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

RTM_SPLAY and RTM_SYNC have no synaptic coupling at all (they're
uncoupled baselines the book compares the coupled cases against), so
there's no network dynamics for Brian2 to add there. The four
sub-examples below all involve real all-to-all excitatory coupling.

Initial conditions for these all place the cells on the RTM neuron's
periodic orbit at a given phase (found by running the single-neuron
model to its 5th spike, then interpolating). That warm-up is a pure
numerical utility, not something Brian2 changes -- so the phase values
below are precomputed once with a plain `scipy.odeint` run and hard-coded
here, matching the book's own `rtm_init`/`splayState` helpers.

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np

### RTM Network with All-to-All Fast Excitation

`g_syn` can be a scalar (uniform coupling) or an (N,N) array
(heterogeneous per-pair coupling, diagonal ignored).

In [ ]:
def simulate_RTM_network(i_ext, g_syn, v0, h0, n0, simulation_time,
                          tau_r=0.5 * b2.ms, tau_d=2 * b2.ms, tau_dq=0.2097 * b2.ms,
                          dt=0.01 * b2.ms):
    N = len(v0)
    El = -67 * b2.mV
    EK = -100 * b2.mV
    ENa = 50 * b2.mV
    gl = 0.1 * b2.msiemens
    gK = 80 * b2.msiemens
    gNa = 100 * b2.msiemens
    C = 1 * b2.ufarad

    eqs = """
    I_e : amp
    Isyn : amp

    alphah = 0.128 * exp(-(vm + 50.0*mV) / (18.0*mV))/ms :Hz
    alpham = 0.32/mV * (vm + 54*mV) / (1.0 - exp(-(vm + 54.0*mV) / (4.0*mV)))/ms:Hz
    alphan = 0.032/mV * (vm + 52*mV) / (1.0 - exp(-(vm + 52.0*mV) / (5.0*mV)))/ms:Hz

    betah  = 4.0 / (1.0 + exp(-(vm + 27.0*mV) / (5.0*mV)))/ms:Hz
    betam  = 0.28/mV * (vm + 27.0*mV) / (exp((vm + 27.0*mV) / (5.0*mV)) - 1.0)/ms:Hz
    betan  = 0.5 * exp(-(vm + 57.0*mV) / (40.0*mV))/ms:Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + Isyn + gNa*m**3*h*(ENa-vm) + \
        gl*(El-vm) + gK*n**4*(EK-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dq/dt = 5/ms*(1+tanh(vm/(10*mV)))*(1-q) - q/tau_dq : 1
    ds/dt = q*(1-s)/tau_r - s/tau_d : 1
    dvm/dt = membrane_Im/C : volt
    """

    neurons = b2.NeuronGroup(N, eqs, method="rk4", dt=dt,
                             namespace={"tau_r": tau_r, "tau_d": tau_d, "tau_dq": tau_dq})
    neurons.vm = np.asarray(v0) * b2.mV
    neurons.h = np.asarray(h0)
    neurons.n = np.asarray(n0)
    neurons.I_e = np.broadcast_to(i_ext, (N,)) * b2.uA
    neurons.q = 0
    neurons.s = 0

    syn = b2.Synapses(neurons, neurons, "w : siemens\nIsyn_post = -w*s_pre*vm_post : amp (summed)")
    syn.connect(condition="i!=j")
    g_syn_arr = np.broadcast_to(g_syn, (N, N))
    syn.w = g_syn_arr[syn.i[:], syn.j[:]] * b2.msiemens

    st_mon = b2.StateMonitor(neurons, "vm", record=True)
    net = b2.Network(neurons, syn, st_mon)
    net.run(simulation_time)
    return st_mon


def spike_times_from_trace(t, v, threshold=-20):
    idx = np.where((v[:-1] >= threshold) & (v[1:] < threshold))[0]
    v_pre, v_post = v[idx], v[idx + 1]
    t_pre, t_post = t[idx], t[idx + 1]
    return (t_pre * (-v_post - threshold) + t_post * (threshold + v_pre)) / (v_pre - v_post)

### Figure 24.2
Two RTM Cells, Perturbed Out of Synchrony and Recovering

Both cells start on the limit cycle at the same phase (already
near-synchronous); after the 10th spike, cell 1 is nudged by -1e-5mV to
show the synchronous state is attracting.

In [ ]:
ic_v, ic_h, ic_n = -68.7035636, 0.9980982, 0.0251301

# Reimplemented inline (rather than via simulate_RTM_network) so the
# simulation can be paused after the 10th spike to apply the book's
# perturbation (v[0] -= 1e-5 mV), then resumed.
El = -67 * b2.mV
EK = -100 * b2.mV
ENa = 50 * b2.mV
gl = 0.1 * b2.msiemens
gK = 80 * b2.msiemens
gNa = 100 * b2.msiemens
C = 1 * b2.ufarad
tau_r, tau_d, tau_dq = 0.5 * b2.ms, 2 * b2.ms, 0.2097 * b2.ms

eqs = """
I_e : amp
Isyn : amp

alphah = 0.128 * exp(-(vm + 50.0*mV) / (18.0*mV))/ms :Hz
alpham = 0.32/mV * (vm + 54*mV) / (1.0 - exp(-(vm + 54.0*mV) / (4.0*mV)))/ms:Hz
alphan = 0.032/mV * (vm + 52*mV) / (1.0 - exp(-(vm + 52.0*mV) / (5.0*mV)))/ms:Hz

betah  = 4.0 / (1.0 + exp(-(vm + 27.0*mV) / (5.0*mV)))/ms:Hz
betam  = 0.28/mV * (vm + 27.0*mV) / (exp((vm + 27.0*mV) / (5.0*mV)) - 1.0)/ms:Hz
betan  = 0.5 * exp(-(vm + 57.0*mV) / (40.0*mV))/ms:Hz

m = alpham / (alpham + betam) : 1
membrane_Im = I_e + Isyn + gNa*m**3*h*(ENa-vm) + \
    gl*(El-vm) + gK*n**4*(EK-vm) : amp

dn/dt = alphan*(1-n)-betan*n : 1
dh/dt = alphah*(1-h)-betah*h : 1
dq/dt = 5/ms*(1+tanh(vm/(10*mV)))*(1-q) - q/tau_dq : 1
ds/dt = q*(1-s)/tau_r - s/tau_d : 1
dvm/dt = membrane_Im/C : volt
"""

neurons = b2.NeuronGroup(2, eqs, threshold="vm>-20*mV", reset="", refractory=3*b2.ms,
                         method="rk4", dt=0.01 * b2.ms,
                         namespace={"tau_r": tau_r, "tau_d": tau_d, "tau_dq": tau_dq})
neurons.vm = [ic_v, ic_v] * b2.mV
neurons.h = [ic_h, ic_h]
neurons.n = [ic_n, ic_n]
neurons.I_e = 0.3 * b2.uA
neurons.q = 0
neurons.s = 0

syn2 = b2.Synapses(neurons, neurons, "w : siemens\nIsyn_post = -w*s_pre*vm_post : amp (summed)")
syn2.connect(condition="i!=j")
syn2.w = 0.0075 * 29 * b2.msiemens

st_mon2 = b2.StateMonitor(neurons, "vm", record=True)
sp_mon2 = b2.SpikeMonitor(neurons, variables="vm")
net2 = b2.Network(neurons, syn2, st_mon2, sp_mon2)

# run in short increments until cell 0 has fired its 10th spike, then
# perturb and run the remainder in one go
perturbed = False
while net2.t < 1000 * b2.ms:
    net2.run(5 * b2.ms)
    if not perturbed and np.sum(sp_mon2.i == 0) >= 10:
        neurons.vm[0] -= 1e-5 * b2.mV
        perturbed = True
net2.run(0 * b2.ms)

t1 = spike_times_from_trace(st_mon2.t / b2.ms, st_mon2.vm[0] / b2.mV)
t2 = spike_times_from_trace(st_mon2.t / b2.ms, st_mon2.vm[1] / b2.mV)
n_spk = min(len(t1), len(t2))
diff = t2[:n_spk] - t1[:n_spk]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(np.arange(1, n_spk + 1), diff, ".k", markersize=15)
ax.set_xlabel("spike #")
ax.set_title("spike time difference [ms]")
ax.set_xlim(0, 25)
ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

### Figure 24.3
N=30 RTM Cells Starting from a Splay State, Synchronizing via Fast Excitation

In [ ]:
splay_v = [-57.96487771711115, -60.67529477622564, -61.66859439893292, -62.28949662012294, -62.75768664314031, -63.149016607556185, -63.49902603008257, -63.82794127676847, -64.14920340635686, -64.47307116099483, -64.80842679780842, -65.16383704092124, -65.54828067462043, -65.97174001458585, -66.44574323256687, -66.98388087443988, -67.60234685575026, -68.32043941039029, -69.16109122119718, -70.1513332760955, -71.32283124893166, -72.71255070389026, -74.36360987419759, -76.32657798921922, -78.66100472793359, -81.4374626383642, -84.7396973066464, -88.66727241394727, -93.33781041466597, -95.88573175428836]
splay_h = [0.981814476441736, 0.9859047906132945, 0.9877926835876856, 0.989059269401747, 0.9900384598634814, 0.9908585567006193, 0.9915824420885953, 0.9922455842257107, 0.9928698598682429, 0.9934694775402727, 0.9940537950735784, 0.9946286920919714, 0.9951972247694668, 0.9957598836908329, 0.996314657517932, 0.9968570506680346, 0.9973801909465683, 0.9978751826681808, 0.9983318100144289, 0.9987396674201263, 0.999089587096327, 0.9993750305825367, 0.9995927426482656, 0.9997409993562579, 0.9998101830856192, 0.9997400418865574, 0.9991806415734541, 0.9956393321018017, 0.9621281778442009, 0.36348072715737517]
splay_n = [0.10600388413604728, 0.08739450857375754, 0.0790043443109981, 0.07344906895844908, 0.06916843385277119, 0.06556869603290484, 0.06235837981480425, 0.05936953345262245, 0.05649309518463365, 0.053651119587613295, 0.05078344375915591, 0.04784108970867037, 0.044783349581920935, 0.04157730629123403, 0.038199324453259215, 0.03463837331985402, 0.03090079570829818, 0.027016045527197698, 0.023041926266107824, 0.019067432007974082, 0.015210506252305517, 0.011608657960724942, 0.008401895428280638, 0.005711720652281348, 0.0036289317794473913, 0.002262773931443455, 0.0021457113567593805, 0.007202963877950449, 0.05126971132931858, 0.5017407889157794]

sm3 = simulate_RTM_network(0.3, 0.0075, splay_v, splay_h, splay_n, 200 * b2.ms)

fig, ax = plt.subplots(figsize=(8, 6))
for i in range(len(splay_v)):
    st = spike_times_from_trace(sm3.t / b2.ms, sm3.vm[i] / b2.mV)
    ax.plot(st, [i + 1] * len(st), ".k", markersize=4)
ax.set_xlabel("t [ms]")
ax.set_ylabel("neuron #")
ax.set_xlim(0, 200)
plt.tight_layout()
plt.show()

### Figure 24.4
N=30 RTM Cells, Long Run from a Splay State (Raster + Locked Frequency)

In [ ]:
sm4 = simulate_RTM_network(0.3, 0.0075, splay_v, splay_h, splay_n, 10000 * b2.ms)

t_full = sm4.t / b2.ms
window = t_full >= t_full[-1] - 200

fig, ax = plt.subplots(figsize=(8, 6))
spikes_last = None
for i in range(len(splay_v)):
    st = spike_times_from_trace(t_full, sm4.vm[i] / b2.mV)
    st = st[st >= t_full[-1] - 200]
    ax.plot(st, [i + 1] * len(st), ".k", markersize=4)
    if i == 0:
        st1_full = spike_times_from_trace(t_full, sm4.vm[i] / b2.mV)
ax.set_xlabel("t [ms]")
ax.set_ylabel("neuron #")
plt.tight_layout()
plt.show()

frequency = 1000.0 / (st1_full[-1] - st1_full[-2])
print("locked frequency [Hz]:", frequency)

### Figure 24.5
N=30 RTM Cells with Heterogeneous Drive and Coupling

`i_ext` and the pairwise `g_syn` are drawn randomly (as in the book's
script -- exact random draws aren't meant to match, only the qualitative
result that the network still locks to a common frequency despite the
heterogeneity).

In [ ]:
rng = np.random.default_rng(63806)
N = 30
i_ext_het = 0.25 + rng.random(N) * 0.1
g_syn_het = 0.00625 + rng.random((N, N)) * 0.0025
np.fill_diagonal(g_syn_het, 0.0)
phases_het = rng.random(N)

# reuse the splay-state trajectory (same intrinsic model, i_ext=0.3) to
# get a point on the limit cycle near each neuron's own drive; since the
# book's own point here is qualitative (does a heterogeneous network
# still lock?), starting near-but-not-exactly on each neuron's own
# orbit is enough to reach the same locked state after the transient
v0_het = np.interp(phases_het, np.linspace(0, 1, len(splay_v)), sorted(splay_v))
h0_het = np.interp(phases_het, np.linspace(0, 1, len(splay_h)), sorted(splay_h))
n0_het = np.interp(phases_het, np.linspace(0, 1, len(splay_n)), sorted(splay_n))

sm5 = simulate_RTM_network(i_ext_het, g_syn_het, v0_het, h0_het, n0_het, 200 * b2.ms)

fig, ax = plt.subplots(figsize=(8, 6))
for i in range(N):
    st = spike_times_from_trace(sm5.t / b2.ms, sm5.vm[i] / b2.mV)
    ax.plot(st, [i + 1] * len(st), ".k", markersize=4)
ax.set_xlabel("t [ms]")
ax.set_ylabel("neuron #")
plt.tight_layout()
plt.show()